# 01 — Макро-модуль
Анализ и прогнозирование макро-факторов.

**Разделы:**
1. Настройка
2. История макро-факторов
3. Запуск макро-прогноза (VECM + MR + EWA)
4. Анализ прогнозов
5. Revenue регрессия на макро-факторы
6. Сравнение методов прогнозирования HRC

## §1 Настройка

In [ ]:
import sys, math
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID    = 'us_steel'
SCENARIO_NAME = 'base'
DB_PATH       = ROOT / 'data_mart_v2.db'
MACRO_CONFIG  = ROOT / 'companies' / COMPANY_ID / 'configs' / 'macro_ecm.yaml'

import logging, pandas as pd
logging.basicConfig(level=logging.WARNING)

from engine.database.repository import Repository
print(f'DB: {DB_PATH.name} exists={DB_PATH.exists()}')

## §2 История макро-факторов

In [ ]:
with Repository(db_path=DB_PATH) as repo:
    rows = repo.query('SELECT DISTINCT factor_name FROM macro_factors ORDER BY factor_name')
    factors = [r['factor_name'] for r in rows]
    print(f'Факторов в БД: {len(factors)}')
    print()
    
    # Таблица истории ключевых факторов
    KEY_FACTORS = ['steel_price_hrc', 'steel_ppi_iron_steel', 'brent', 'gdp_us', 'cpi_us']
    
    hist_data = {}
    all_years = set()
    for f in KEY_FACTORS:
        s = repo.get_macro_factor(f)
        hist_data[f] = s
        all_years.update(s.keys())
    
    years = sorted(all_years)
    rows_table = []
    for yr in years[-10:]:  # последние 10 лет
        row = {'Год': yr}
        for f in KEY_FACTORS:
            v = hist_data[f].get(yr)
            row[f] = f'{v:.0f}' if v else 'N/A'
        rows_table.append(row)
    
    df = pd.DataFrame(rows_table).set_index('Год')
    print('История ключевых факторов:')
    display(df)

## §3 Запуск макро-прогноза

In [ ]:
from engine.macro.runner import run_macro

with Repository(db_path=DB_PATH) as repo:
    # Очищаем старые прогнозы
    sid = repo.get_scenario_id(COMPANY_ID, SCENARIO_NAME)
    if sid:
        repo.execute('DELETE FROM macro_forecasts WHERE company_id=? AND scenario_id=?',
                    (COMPANY_ID, sid))
    
    macro_result = run_macro(
        company_id=COMPANY_ID,
        repo=repo,
        config_path=MACRO_CONFIG,
        scenario_name=SCENARIO_NAME,
        forecast_years=5,
    )
    
    print(macro_result.summary())
    print()
    print('Методы прогнозирования:')
    methods = {}
    for f, m in macro_result.methods_used.items():
        methods.setdefault(m, []).append(f)
    for method, factors in methods.items():
        print(f'  {method}: {factors}')

## §4 Анализ прогнозов

In [ ]:
with Repository(db_path=DB_PATH) as repo:
    sid = repo.get_scenario_id(COMPANY_ID, SCENARIO_NAME)
    forecasts = repo.get_macro_forecasts(COMPANY_ID, sid)
    
    KEY_FACTORS = ['steel_price_hrc', 'steel_ppi_iron_steel', 'brent', 'gdp_us', 'cpi_us']
    
    rows = []
    for f in KEY_FACTORS:
        if f not in forecasts:
            continue
        hist = repo.get_macro_factor(f)
        fc   = forecasts[f]
        last_hist = hist.get(max(hist.keys())) if hist else None
        
        row = {'Фактор': f, 'Last hist': f'{last_hist:.0f}' if last_hist else 'N/A'}
        for yr in range(2025, 2030):
            v = fc.get(yr)
            row[yr] = f'{v:.0f}' if v else 'N/A'
        if last_hist and fc.get(2025):
            chg = (fc[2025]/last_hist - 1)*100
            row['Δ2025%'] = f'{chg:+.1f}%'
        rows.append(row)
    
    df = pd.DataFrame(rows).set_index('Фактор')
    print('Прогнозы макро-факторов 2025-2029:')
    display(df)

## §5 Revenue ~ HRC регрессия

In [ ]:
from engine.model.revenue_models import ols_single_factor

with Repository(db_path=DB_PATH) as repo:
    rev_series = repo.get_metric_series(COMPANY_ID, 'IS', 'revenue')
    hrc_series = repo.get_macro_factor('steel_price_hrc')
    sid = repo.get_scenario_id(COMPANY_ID, SCENARIO_NAME)
    fc = repo.get_macro_forecasts(COMPANY_ID, sid)
    hrc_fc = fc.get('steel_price_hrc', {})
    
    forecast_years = list(range(2025, 2030))
    fc_rev, beta, r2 = ols_single_factor(
        rev_series, hrc_series, hrc_fc, forecast_years,
        use_dln=True, chainlink=True,
    )
    
    print(f'Revenue ~ HRC OLS:')
    print(f'  beta = {beta:.4f}  R² = {r2:.4f}')
    print(f'  Интерпретация: HRC +10% → Revenue {beta*10:+.1f}%')
    print()
    print(f'  {"Год":<6} {"HRC":>8} {"Rev forecast $B":>16}')
    for yr in forecast_years:
        hrc_val = hrc_fc.get(yr, 0)
        rev_val = fc_rev.get(yr, 0)
        print(f'  {yr:<6} {hrc_val:>8.0f} {rev_val/1e9:>16.2f}')

## §6 Сравнение методов прогноза HRC

In [ ]:
from engine.macro.commodity_models import mean_reversion_forecast, rw_drift_clamped
from engine.macro.runner import _ewa_forecast

with Repository(db_path=DB_PATH) as repo:
    hrc = repo.get_macro_factor('steel_price_hrc')
    
    vals = sorted(hrc.values())
    n = len(vals)
    median = (vals[n//2-1]+vals[n//2])/2 if n%2==0 else vals[n//2]
    
    print(f'HRC статистика:')
    print(f'  Min={min(vals):.0f}  P25={vals[n//4]:.0f}  Median={median:.0f}  P75={vals[3*n//4]:.0f}  Max={max(vals):.0f}')
    print(f'  Last (2024): {hrc.get(2024, hrc[max(hrc.keys())]):.0f}')
    print()
    
    methods = {
        'EWA (hl=3)':   _ewa_forecast(hrc, 5, halflife=3.0),
        'MR kappa=0.15':mean_reversion_forecast(hrc, 5, kappa=0.15),
        'MR kappa=0.30':mean_reversion_forecast(hrc, 5, kappa=0.30),
        'MR kappa=0.50':mean_reversion_forecast(hrc, 5, kappa=0.50),
        'RW Drift':     rw_drift_clamped(hrc, 5, ewa_halflife=5.0),
    }
    
    rows = []
    for method, fc in methods.items():
        row = {'Метод': method}
        for yr in range(2025, 2030):
            row[yr] = f'{fc.get(yr,0):.0f}'
        rows.append(row)
    
    df = pd.DataFrame(rows).set_index('Метод')
    print('Прогнозы HRC по методам ($/ short ton):')
    display(df)
    print()
    print('S&P Global консенсус 2025: ~$748-778')
    print(f'Долгосрочная медиана: ${median:.0f}')